In [1]:
import os
import time
import pandas as pd
from tqdm import tqdm
from deep_translator import GoogleTranslator

# ============================================================
# KONFIGURASI
# ============================================================
# Ganti nama file input dengan nama file CSV hasil generate Anda
INPUT_CSV  = "/mnt/extended-home/dzakaaufa/dataset/caption/train_data_inject_split.csv"
# File output akhir
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/dataset/caption/train_data_indo.csv"

# ============================================================
# FUNGSI TRANSLASI DENGAN RETRY
# ============================================================
def translate_text(text, translator, max_retries=3):
    if not isinstance(text, str) or not text.strip():
        return ""
        
    for attempt in range(max_retries):
        try:
            # Beri sedikit jeda agar tidak di-banned oleh Google
            time.sleep(0.5) 
            translated = translator.translate(text)
            return translated
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2) # Jeda lebih lama jika error sebelum mencoba lagi
                continue
            else:
                print(f"\n[ERROR] Gagal menerjemahkan: {e}")
                return text # Kembalikan teks asli jika gagal total

# ============================================================
# MAIN
# ============================================================
def main():
    print("Membaca dataset...")
    df = pd.read_csv(INPUT_CSV)
    
    # Di skrip generate Anda sebelumnya, kolom caption bernama 'CAPTION_ID' meski isinya Inggris.
    # Kita ubah namanya agar tidak bingung.
    if 'caption_en' in df.columns and 'caption_en' not in df.columns:
        df.rename(columns={'caption_en': 'caption_en'}, inplace=True)
        
    if 'caption_en' not in df.columns:
        raise ValueError("Kolom 'caption_en' (atau 'CAPTION_ID' dari skrip lama) tidak ditemukan!")

    # Inisiasi Translator (Dari Inggris ke Indonesia)
    translator = GoogleTranslator(source='en', target='id')

    # Buat kolom baru jika belum ada
    if 'CAPTION_ID' not in df.columns:
        df['CAPTION_ID'] = None

    total_rows = len(df)
    
    # Hitung yang belum ditranslasi (untuk melanjutkan jika terhenti)
    untranslated_mask = df['CAPTION_ID'].isna() | (df['CAPTION_ID'] == "")
    untranslated_count = untranslated_mask.sum()
    
    print(f"Total baris: {total_rows}")
    print(f"Sisa yang perlu ditranslasi: {untranslated_count}\n")

    # Iterasi hanya pada baris yang belum memiliki translasi
    for index, row in tqdm(df[untranslated_mask].iterrows(), total=untranslated_count, desc="Translating"):
        english_text = row['caption_en']
        
        indonesian_text = translate_text(english_text, translator)
        df.at[index, 'CAPTION_ID'] = indonesian_text
        
        # Auto-save setiap 20 baris
        if (index + 1) % 20 == 0:
            df.to_csv(OUTPUT_CSV, index=False)

    # Final Save
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Translasi Selesai! Disimpan di: {OUTPUT_CSV}")
    
    print("\n--- Preview Hasil ---")
    for _, row in df.head(2).iterrows():
        print(f"\n[EN]: {row['caption_en'][:150]}...")
        print(f"[ID]: {row['CAPTION_ID'][:150]}...")

if __name__ == "__main__":
    main()

Membaca dataset...
Total baris: 3327
Sisa yang perlu ditranslasi: 3327



Translating: 100%|██████████| 3327/3327 [1:25:19<00:00,  1.54s/it]


✅ Translasi Selesai! Disimpan di: /mnt/extended-home/dzakaaufa/dataset/caption/train_data_indo.csv

--- Preview Hasil ---

[EN]: The batik fabric features the wahyu_tumurun motif, characterized by a golden-yellow background and dark green motifs. The main motifs include stylized...
[ID]: Kain batik ini menampilkan motif wahyu_tumurun dengan ciri khas latar belakang kuning keemasan dan motif hijau tua. Motif utamanya meliputi bunga dan ...

[EN]: The batik fabric features the liong motif, characterized by a deep red background and vibrant blue, yellow, pink, and white motifs. The main motifs in...
[ID]: Kain batiknya menampilkan motif liong yang bercirikan latar belakang merah tua dan motif biru cerah, kuning, merah jambu, dan putih. Motif utamanya me...
